# Asset Value Analysis
Analyse Merton-implied asset values, market capitalisation, and total liabilities across firms matched to CDS market data for 1Y, 3Y, and 5Y maturities.

### Imports & Configuration

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import src modules
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils import config

In [ ]:
merged_data_path = config.OUTPUT_DIR / 'merged_data_with_merton.csv'
df_merged = pd.read_csv(merged_data_path, parse_dates=['date'])
df_merged.head()

### Load Merton Data & Market CDS Spreads

In [ ]:
def load_cds_market_data(filepath, maturity):
    # Parse raw CDS Excel, company names in row 3, data starts at row 5
    df = pd.read_excel(filepath, header=None)
    
    company_names_raw = df.iloc[3, 2:].tolist()
    company_names = []
    for name in company_names_raw:
        if pd.notna(name):
            clean = name.split(' SNR ')[0] if ' SNR ' in str(name) else str(name)
            company_names.append(clean)
        else:
            company_names.append(None)
    
    dates = pd.to_datetime(df.iloc[5:, 0], errors='coerce')
    data = df.iloc[5:, 2:].copy()
    data.columns = company_names[:len(data.columns)]
    data['date'] = dates.values
    
    # Reshape from wide to long format
    data_long = data.melt(
        id_vars=['date'], 
        var_name='company_cds', 
        value_name=f'cds_market_{maturity}y_bps'
    )
    data_long[f'cds_market_{maturity}y_bps'] = pd.to_numeric(
        data_long[f'cds_market_{maturity}y_bps'], errors='coerce'
    )
    
    return data_long.dropna(subset=['date', 'company_cds'])

# Map CDS company names to model company names
COMPANY_MAPPING = {
    'ADIDAS AG': 'ADIDAS AG',
    'AIR LIQUIDE SA': "L'AIR LIQUIDE SA",
    'AIRBUS SE': 'AIRBUS SE',
    'ALLIANZ SE': 'ALLIANZ SE',
    'ANHEUSER-BUSCH INBE': 'ANHEUSER-BUSCH INBEV',
    'AXA': 'AXA SA',
    'BASF SE': 'BASF SE',
    'BAYER AG': 'BAYER AG',
    'BAYER MOTOREN WERKE': 'BAYERISCHE MOTOREN WERKE AKT',
    'BNP PARIBAS SNRFOR MM14 E - CDS PREM. MID': 'BNP PARIBAS',
    'DANONE SA': 'DANONE SA',
    'DEUTSCHE POST AG': 'DEUTSCHE POST AG',
    'DEUTSCHE TELEKOM AG': 'DEUTSCHE TELEKOM',
    'ENEL S.P.A.': 'ENEL SPA',
    'ENI S.P.A.': 'ENI SPA',
    'IBERDROLA, S.A.': 'IBERDROLA SA',
    'INFINEON TECS': 'INFINEON TECHNOLOGIES AG',
    'ING GROEP N.V.': 'ING GROEP NV',
    'INTESA SANPAOLO SPA': 'INTESA SANPAOLO SPA',
    'KERING SA': 'KERING SA',
    'KON AHOLD DELHAIZE': 'KONINKLIJKE AHOLD DELHAIZE',
    "L'OREAL": 'LOREAL SA',
    'LVMH MOET HENNESSY': 'LVMH MOET HENNESSY LOUIS V',
    'MUNICH REINSURANCE': 'MUNICH RE CO',
    'NOKIA OYJ': 'NOKIA OYJ',
    'ORANGE S.A.': 'ORANGE SA',
    'SANOFI SA': 'SANOFI',
    'SAP SE': 'SAP SE',
    'SCHNEIDER ELECTRIC': 'SCHNEIDER ELECTRIC S E',
    'SIEMENS AG': 'SIEMENS AG',
    'TOTALENERGIES SE': 'TOTALENERGIES SE',
    'UNICREDIT SPA': 'UNICREDIT SPA',
    'VINCI': 'VINCI SA',
    'WOLTERS KLUWER NV': 'WOLTERS KLUWER NV',
}

# Load market CDS spreads for all three maturities
input_dir = config.DATA_DIR / 'input'
cds_1y = load_cds_market_data(input_dir / 'CDS_1y_mat_data.xlsx', 1)
cds_3y = load_cds_market_data(input_dir / 'CDS_3y_mat_data.xlsx', 3)
cds_5y = load_cds_market_data(input_dir / 'CDS_5y_mat_data.xlsx', 5)

# Merge all maturities and map to gvkey identifiers
market_cds = cds_1y.merge(cds_3y, on=['date', 'company_cds'], how='outer')
market_cds = market_cds.merge(cds_5y, on=['date', 'company_cds'], how='outer')
market_cds['company'] = market_cds['company_cds'].map(COMPANY_MAPPING)

company_to_gvkey = df_merged[['company', 'gvkey']].drop_duplicates().set_index('company')['gvkey'].to_dict()
market_cds['gvkey'] = market_cds['company'].map(company_to_gvkey)
market_cds = market_cds.dropna(subset=['gvkey'])
market_cds['gvkey'] = market_cds['gvkey'].astype(int)

In [ ]:
def get_firms_with_cds_data(date, maturity):
    # Return the set of gvkeys that have non-missing CDS data on a given date
    col_name = f'cds_market_{maturity}y_bps'
    date_data = market_cds[market_cds['date'] == pd.Timestamp(date)]
    firms_with_data = date_data[date_data[col_name].notna()]['gvkey'].unique()
    return set(firms_with_data)

## Asset Value Analysis by CDS Maturity

In [ ]:
def analyze_asset_values_matching_cds_firms(maturity=5, start_date='2017-06-01'):
    # Compute daily asset value stats for firms with CDS data and plot time series
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    
    results = []
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['asset_value'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        asset_values_bn = date_data['asset_value'] / 1e9
        results.append({
            'date': date,
            'num_firms': len(cds_firms),
            'num_firms_with_asset': len(date_data),
            'mean': asset_values_bn.mean(),
            'median': asset_values_bn.median(),
            'q25': asset_values_bn.quantile(0.25),
            'q75': asset_values_bn.quantile(0.75),
            'std': asset_values_bn.std()
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    fig.suptitle(f'Asset Value Analysis - Matching {maturity}Y CDS Firms', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    axes[0].plot(daily_stats['date'], daily_stats['num_firms'], linewidth=2, color='blue', label='Firms with CDS data')
    axes[0].set_title(f'Number of Firms in Dataset Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Firms', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
    axes[0].legend()
    
    axes[1].plot(daily_stats['date'], daily_stats['mean'], linewidth=2, label='Mean', color='orange', alpha=0.8)
    axes[1].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue', alpha=0.8)
    axes[1].set_title(f'Mean vs Median Asset Values ({maturity}Y)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Asset Value (billions)', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    axes[2].fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], 
                         alpha=0.3, color='lightblue', label='25th-75th percentile')
    axes[2].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue')
    axes[2].set_title(f'Asset Value Distribution (25th, 50th, 75th percentiles)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Date', fontsize=12)
    axes[2].set_ylabel('Asset Value (billions)', fontsize=12)
    axes[2].legend(fontsize=11)
    axes[2].grid(True, alpha=0.3)
    axes[2].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return daily_stats

In [ ]:
def plot_asset_value_distribution(maturity=5, start_date='2017-06-01'):
    # Plot asset value percentiles over time and print summary stats for key periods
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    
    results = []
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['asset_value'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        asset_values_bn = date_data['asset_value'] / 1e9
        results.append({
            'date': date,
            'mean': asset_values_bn.mean(),
            'median': asset_values_bn.median(),
            'q25': asset_values_bn.quantile(0.25),
            'q75': asset_values_bn.quantile(0.75)
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.plot(daily_stats['date'], daily_stats['mean'], linewidth=2.5, label='Mean', color='#d62728', alpha=0.9)
    ax.plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='#1f77b4', alpha=0.8)
    ax.plot(daily_stats['date'], daily_stats['q75'], linewidth=1.5, label='75th percentile', color='#2ca02c', alpha=0.7, linestyle='--')
    ax.plot(daily_stats['date'], daily_stats['q25'], linewidth=1.5, label='25th percentile', color='#ff7f0e', alpha=0.7, linestyle='--')
    ax.fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], alpha=0.15, color='gray', label='IQR')
    ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle=':', alpha=0.5, linewidth=2, label='COVID-19 Start')
    
    ax.set_title(f'Asset Value Distribution Over Time ({maturity}Y)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Asset Value (billions)', fontsize=12)
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"SUMMARY STATISTICS: Asset Values - {maturity}Y Maturity")
    
    periods = [
        ('2020-03-01', '2020-05-31', 'COVID Peak (Mar-May 2020)'),
        ('2023-01-01', '2023-03-31', 'Banking Crisis (Jan-Mar 2023)'),
        ('2024-01-01', '2024-12-31', 'Recent Period (2024)'),
    ]
    
    print(f"\n{'Period':<30} {'Mean':<12} {'Median':<12} {'25th %':<12} {'75th %':<12} {'Spread':<12}")
    
    for start, end, label in periods:
        period_data = daily_stats[(daily_stats['date'] >= start) & (daily_stats['date'] <= end)]
        if not period_data.empty:
            print(f"{label:<30} {period_data['mean'].mean():>11.2f}  {period_data['median'].mean():>11.2f}  {period_data['q25'].mean():>11.2f}  {period_data['q75'].mean():>11.2f}  {period_data['q75'].mean()-period_data['q25'].mean():>11.2f}")
    
    return daily_stats

### 5-Year Maturity

In [ ]:
stats_5y = analyze_asset_values_matching_cds_firms(maturity=5)

In [ ]:
dist_stats_5y = plot_asset_value_distribution(maturity=5)

### 3-Year Maturity

In [ ]:
stats_3y = analyze_asset_values_matching_cds_firms(maturity=3)

In [ ]:
dist_stats_3y = plot_asset_value_distribution(maturity=3)

### 1-Year Maturity

In [ ]:
stats_1y = analyze_asset_values_matching_cds_firms(maturity=1)

In [ ]:
dist_stats_1y = plot_asset_value_distribution(maturity=1)

### Cross-Maturity Comparison

In [ ]:
# Compare median asset values across 1Y, 3Y, 5Y maturity datasets
fig, ax = plt.subplots(figsize=(14, 8))

start_date = pd.Timestamp('2017-06-01')

if 'stats_1y' in locals() and not stats_1y.empty:
    filtered_1y = stats_1y[stats_1y['date'] >= start_date]
    ax.plot(filtered_1y['date'], filtered_1y['median'], linewidth=2, label='1Y Maturity', alpha=0.8)
if 'stats_3y' in locals() and not stats_3y.empty:
    filtered_3y = stats_3y[stats_3y['date'] >= start_date]
    ax.plot(filtered_3y['date'], filtered_3y['median'], linewidth=2, label='3Y Maturity', alpha=0.8)
if 'stats_5y' in locals() and not stats_5y.empty:
    filtered_5y = stats_5y[stats_5y['date'] >= start_date]
    ax.plot(filtered_5y['date'], filtered_5y['median'], linewidth=2, label='5Y Maturity', alpha=0.8)

ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
ax.set_title('Median Asset Values Across Different Maturity Datasets', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Median Asset Value (billions)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare firm counts across maturities to check CDS data availability
fig, ax = plt.subplots(figsize=(14, 8))

start_date = pd.Timestamp('2017-06-01')

if 'stats_1y' in locals() and not stats_1y.empty:
    filtered_1y = stats_1y[stats_1y['date'] >= start_date]
    ax.plot(filtered_1y['date'], filtered_1y['num_firms'], linewidth=2, label='1Y Maturity', alpha=0.8)
if 'stats_3y' in locals() and not stats_3y.empty:
    filtered_3y = stats_3y[stats_3y['date'] >= start_date]
    ax.plot(filtered_3y['date'], filtered_3y['num_firms'], linewidth=2, label='3Y Maturity', alpha=0.8)
if 'stats_5y' in locals() and not stats_5y.empty:
    filtered_5y = stats_5y[stats_5y['date'] >= start_date]
    ax.plot(filtered_5y['date'], filtered_5y['num_firms'], linewidth=2, label='5Y Maturity', alpha=0.8)

ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
ax.set_title('Number of Firms with CDS Data by Maturity', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Firms', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Market Capitalisation Analysis

In [ ]:
def analyze_market_cap_matching_cds_firms(maturity=5, start_date='2017-06-01'):
    # Compute daily market cap stats for firms with CDS data and plot time series
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    results = []
    
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['mkt_cap'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        mkt_cap_bn = date_data['mkt_cap'] / 1e9
        results.append({
            'date': date,
            'num_firms': len(cds_firms),
            'num_firms_with_mktcap': len(date_data),
            'mean': mkt_cap_bn.mean(),
            'median': mkt_cap_bn.median(),
            'q25': mkt_cap_bn.quantile(0.25),
            'q75': mkt_cap_bn.quantile(0.75),
            'std': mkt_cap_bn.std()
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    fig.suptitle(f'Market Capitalization Analysis - Matching {maturity}Y CDS Firms', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    axes[0].plot(daily_stats['date'], daily_stats['num_firms'], linewidth=2, color='blue', label='Firms with CDS data')
    axes[0].set_title('Number of Firms in Dataset Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Firms', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
    axes[0].legend()
    
    axes[1].plot(daily_stats['date'], daily_stats['mean'], linewidth=2, label='Mean', color='orange', alpha=0.8)
    axes[1].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue', alpha=0.8)
    axes[1].set_title(f'Mean vs Median Market Cap ({maturity}Y)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Market Cap (billions)', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    axes[2].fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], 
                         alpha=0.3, color='lightblue', label='25th-75th percentile')
    axes[2].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue')
    axes[2].set_title('Market Cap Distribution (25th, 50th, 75th percentiles)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Date', fontsize=12)
    axes[2].set_ylabel('Market Cap (billions)', fontsize=12)
    axes[2].legend(fontsize=11)
    axes[2].grid(True, alpha=0.3)
    axes[2].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return daily_stats

In [ ]:
def plot_market_cap_distribution(maturity=5, start_date='2017-06-01'):
    # Plot market cap percentiles over time and print summary stats for key periods
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    
    results = []
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['mkt_cap'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        mkt_cap_bn = date_data['mkt_cap'] / 1e9
        results.append({
            'date': date,
            'mean': mkt_cap_bn.mean(),
            'median': mkt_cap_bn.median(),
            'q25': mkt_cap_bn.quantile(0.25),
            'q75': mkt_cap_bn.quantile(0.75)
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.plot(daily_stats['date'], daily_stats['mean'], linewidth=2.5, label='Mean', color='#d62728', alpha=0.9)
    ax.plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='#1f77b4', alpha=0.8)
    ax.plot(daily_stats['date'], daily_stats['q75'], linewidth=1.5, label='75th percentile', color='#2ca02c', alpha=0.7, linestyle='--')
    ax.plot(daily_stats['date'], daily_stats['q25'], linewidth=1.5, label='25th percentile', color='#ff7f0e', alpha=0.7, linestyle='--')
    ax.fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], alpha=0.15, color='gray', label='IQR')
    ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle=':', alpha=0.5, linewidth=2, label='COVID-19 Start')
    
    ax.set_title(f'Market Cap Distribution Over Time ({maturity}Y)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Market Cap (billions)', fontsize=12)
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"SUMMARY STATISTICS: Market Capitalization - {maturity}Y Maturity")
    
    periods = [
        ('2020-03-01', '2020-05-31', 'COVID Peak (Mar-May 2020)'),
        ('2023-01-01', '2023-03-31', 'Banking Crisis (Jan-Mar 2023)'),
        ('2024-01-01', '2024-12-31', 'Recent Period (2024)'),
    ]
    
    print(f"\n{'Period':<30} {'Mean':<12} {'Median':<12} {'25th %':<12} {'75th %':<12} {'Spread':<12}")
    print("-"*90)
    
    for start, end, label in periods:
        period_data = daily_stats[(daily_stats['date'] >= start) & (daily_stats['date'] <= end)]
        if not period_data.empty:
            print(f"{label:<30} {period_data['mean'].mean():>11.2f}  {period_data['median'].mean():>11.2f}  {period_data['q25'].mean():>11.2f}  {period_data['q75'].mean():>11.2f}  {period_data['q75'].mean()-period_data['q25'].mean():>11.2f}")
    
    return daily_stats

### 5-Year Maturity

In [ ]:
mktcap_stats_5y = analyze_market_cap_matching_cds_firms(maturity=5)

In [ ]:
mktcap_dist_stats_5y = plot_market_cap_distribution(maturity=5)

### 3-Year Maturity

In [ ]:
mktcap_stats_3y = analyze_market_cap_matching_cds_firms(maturity=3)

In [ ]:
mktcap_dist_stats_3y = plot_market_cap_distribution(maturity=3)

### 1-Year Maturity

In [ ]:
mktcap_stats_1y = analyze_market_cap_matching_cds_firms(maturity=1)

In [ ]:
mktcap_dist_stats_1y = plot_market_cap_distribution(maturity=1)

### Cross-Maturity Comparison

In [ ]:
# Compare median market cap across 1Y, 3Y, 5Y maturity datasets
fig, ax = plt.subplots(figsize=(14, 8))

start_date = pd.Timestamp('2017-06-01')

if 'mktcap_stats_1y' in locals() and not mktcap_stats_1y.empty:
    filtered_1y = mktcap_stats_1y[mktcap_stats_1y['date'] >= start_date]
    ax.plot(filtered_1y['date'], filtered_1y['median'], linewidth=2, label='1Y Maturity', alpha=0.8)
if 'mktcap_stats_3y' in locals() and not mktcap_stats_3y.empty:
    filtered_3y = mktcap_stats_3y[mktcap_stats_3y['date'] >= start_date]
    ax.plot(filtered_3y['date'], filtered_3y['median'], linewidth=2, label='3Y Maturity', alpha=0.8)
if 'mktcap_stats_5y' in locals() and not mktcap_stats_5y.empty:
    filtered_5y = mktcap_stats_5y[mktcap_stats_5y['date'] >= start_date]
    ax.plot(filtered_5y['date'], filtered_5y['median'], linewidth=2, label='5Y Maturity', alpha=0.8)

ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
ax.set_title('Median Market Cap Across Different Maturity Datasets', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Median Market Cap (billions)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Total Liabilities Analysis

In [ ]:
def analyze_liabilities_matching_cds_firms(maturity=5, start_date='2017-06-01'):
    # Compute daily liabilities stats for firms with CDS data and plot time series
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    results = []
    
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['liabilities_total'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        liabilities_bn = date_data['liabilities_total'] / 1e9
        results.append({
            'date': date,
            'num_firms': len(cds_firms),
            'num_firms_with_liab': len(date_data),
            'mean': liabilities_bn.mean(),
            'median': liabilities_bn.median(),
            'q25': liabilities_bn.quantile(0.25),
            'q75': liabilities_bn.quantile(0.75),
            'std': liabilities_bn.std()
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    fig.suptitle(f'Total Liabilities Analysis - Matching {maturity}Y CDS Firms', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    axes[0].plot(daily_stats['date'], daily_stats['num_firms'], linewidth=2, color='blue', label='Firms with CDS data')
    axes[0].set_title('Number of Firms in Dataset Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Firms', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
    axes[0].legend()
    
    axes[1].plot(daily_stats['date'], daily_stats['mean'], linewidth=2, label='Mean', color='orange', alpha=0.8)
    axes[1].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue', alpha=0.8)
    axes[1].set_title(f'Mean vs Median Total Liabilities ({maturity}Y)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Total Liabilities (billions)', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    axes[2].fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], 
                         alpha=0.3, color='lightblue', label='25th-75th percentile')
    axes[2].plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='blue')
    axes[2].set_title('Total Liabilities Distribution (25th, 50th, 75th percentiles)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Date', fontsize=12)
    axes[2].set_ylabel('Total Liabilities (billions)', fontsize=12)
    axes[2].legend(fontsize=11)
    axes[2].grid(True, alpha=0.3)
    axes[2].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return daily_stats

In [ ]:
def plot_liabilities_distribution(maturity=5, start_date='2017-06-01'):
    # Plot liabilities percentiles over time
    all_dates = df_merged[df_merged['date'] >= pd.Timestamp(start_date)]['date'].sort_values().unique()
    
    results = []
    for date in all_dates:
        cds_firms = get_firms_with_cds_data(date, maturity)
        if len(cds_firms) == 0:
            continue
        
        date_data = df_merged[
            (df_merged['date'] == date) & 
            (df_merged['gvkey'].isin(cds_firms)) &
            (df_merged['liabilities_total'].notna())
        ]
        if len(date_data) == 0:
            continue
        
        liabilities_bn = date_data['liabilities_total'] / 1e9
        results.append({
            'date': date,
            'mean': liabilities_bn.mean(),
            'median': liabilities_bn.median(),
            'q25': liabilities_bn.quantile(0.25),
            'q75': liabilities_bn.quantile(0.75)
        })
    
    daily_stats = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.plot(daily_stats['date'], daily_stats['mean'], linewidth=2.5, label='Mean', color='#d62728', alpha=0.9)
    ax.plot(daily_stats['date'], daily_stats['median'], linewidth=2, label='Median', color='#1f77b4', alpha=0.8)
    ax.plot(daily_stats['date'], daily_stats['q75'], linewidth=1.5, label='75th percentile', color='#2ca02c', alpha=0.7, linestyle='--')
    ax.plot(daily_stats['date'], daily_stats['q25'], linewidth=1.5, label='25th percentile', color='#ff7f0e', alpha=0.7, linestyle='--')
    ax.fill_between(daily_stats['date'], daily_stats['q25'], daily_stats['q75'], alpha=0.15, color='gray', label='IQR')
    ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle=':', alpha=0.5, linewidth=2, label='COVID-19 Start')
    
    ax.set_title(f'Total Liabilities Distribution Over Time ({maturity}Y)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Total Liabilities (billions)', fontsize=12)
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return daily_stats

### 5-Year Maturity

In [ ]:
liab_stats_5y = analyze_liabilities_matching_cds_firms(maturity=5)

In [ ]:
liab_dist_stats_5y = plot_liabilities_distribution(maturity=5)

### 3-Year Maturity

In [ ]:
liab_stats_3y = analyze_liabilities_matching_cds_firms(maturity=3)

In [ ]:
liab_dist_stats_3y = plot_liabilities_distribution(maturity=3)

### 1-Year Maturity

In [ ]:
liab_stats_1y = analyze_liabilities_matching_cds_firms(maturity=1)

In [ ]:
liab_dist_stats_1y = plot_liabilities_distribution(maturity=1)

### Cross-Maturity Comparison

In [ ]:
# Compare median liabilities across 1Y, 3Y, 5Y maturity datasets
fig, ax = plt.subplots(figsize=(14, 8))

start_date = pd.Timestamp('2017-06-01')

if 'liab_stats_1y' in locals() and not liab_stats_1y.empty:
    filtered_1y = liab_stats_1y[liab_stats_1y['date'] >= start_date]
    ax.plot(filtered_1y['date'], filtered_1y['median'], linewidth=2, label='1Y Maturity', alpha=0.8)
if 'liab_stats_3y' in locals() and not liab_stats_3y.empty:
    filtered_3y = liab_stats_3y[liab_stats_3y['date'] >= start_date]
    ax.plot(filtered_3y['date'], filtered_3y['median'], linewidth=2, label='3Y Maturity', alpha=0.8)
if 'liab_stats_5y' in locals() and not liab_stats_5y.empty:
    filtered_5y = liab_stats_5y[liab_stats_5y['date'] >= start_date]
    ax.plot(filtered_5y['date'], filtered_5y['median'], linewidth=2, label='5Y Maturity', alpha=0.8)

ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID Start')
ax.set_title('Median Total Liabilities Across Different Maturity Datasets', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Median Total Liabilities (billions)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()